# SageMaker Project Setup

Creates the GitHub repos, CodeConnections connection, and SageMaker Project for the
AutoGluon AutoML CI/CD pipeline (build/train/evaluate/register + real-time and batch deploy).

## Configuration

In [1]:
import json
import subprocess
import time

import boto3
from sagemaker.core.helper.session_helper import Session

REGION = boto3.session.Session().region_name
sess = Session()
BUCKET = sess.default_bucket()

PROJECT_NAME = "autogluon-automl"  # must match ^[a-zA-Z](-*[a-zA-Z0-9])*, max 32 chars
GITHUB_OWNER = subprocess.run(["gh", "api", "user", "--jq", ".login"], capture_output=True, text=True, check=True).stdout.strip()
BUILD_REPO = f"{GITHUB_OWNER}/{PROJECT_NAME}-build"
DEPLOY_REPO = f"{GITHUB_OWNER}/{PROJECT_NAME}-deploy"

sm_client = boto3.client("sagemaker", region_name=REGION)
codeconnections_client = boto3.client("codeconnections", region_name=REGION)

print(f"Region: {REGION}")
print(f"Bucket: {BUCKET}")
print(f"Build repo:  {BUILD_REPO}")
print(f"Deploy repo: {DEPLOY_REPO}")

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/dggallit/Library/Application Support/sagemaker/config.yaml


Region: us-east-1
Bucket: sagemaker-us-east-1-859755744029
Build repo:  dgallitelli/autogluon-automl-build
Deploy repo: dgallitelli/autogluon-automl-deploy


## Create GitHub repos and push seed code

In [2]:
import os

for repo, seed_dir in [(BUILD_REPO, "../seed-code/build"), (DEPLOY_REPO, "../seed-code/deploy")]:
    subprocess.run(["gh", "repo", "create", repo, "--private", "--confirm"], check=True)
    print(f"Created {repo}")

    tmp_clone = f"/tmp/{repo.split('/')[-1]}"
    subprocess.run(["rm", "-rf", tmp_clone])
    subprocess.run(["git", "clone", f"https://github.com/{repo}.git", tmp_clone], check=True)
    subprocess.run(f"cp -r {seed_dir}/* {tmp_clone}/", shell=True, check=True)
    subprocess.run(["git", "-C", tmp_clone, "add", "-A"], check=True)
    subprocess.run(["git", "-C", tmp_clone, "commit", "-m", "Initial seed code"], check=True)
    subprocess.run(["git", "-C", tmp_clone, "push", "origin", "main"], check=True)
    print(f"Pushed seed code to {repo}")

Flag --confirm has been deprecated, Pass any argument to skip confirmation prompt


https://github.com/dgallitelli/autogluon-automl-build
Created dgallitelli/autogluon-automl-build


Cloning into '/tmp/autogluon-automl-build'...


[main (root-commit) ccb0294] Initial seed code
 54 files changed, 1724 insertions(+)
 create mode 100644 automl_pipelines.egg-info/PKG-INFO
 create mode 100644 automl_pipelines.egg-info/SOURCES.txt
 create mode 100644 automl_pipelines.egg-info/dependency_links.txt
 create mode 100644 automl_pipelines.egg-info/entry_points.txt
 create mode 100644 automl_pipelines.egg-info/requires.txt
 create mode 100644 automl_pipelines.egg-info/top_level.txt
 create mode 100644 build/lib/pipelines/__init__.py
 create mode 100644 build/lib/pipelines/__version__.py
 create mode 100644 build/lib/pipelines/_utils.py
 create mode 100644 build/lib/pipelines/automl/__init__.py
 create mode 100644 build/lib/pipelines/automl/evaluate.py
 create mode 100644 build/lib/pipelines/automl/pipeline.py
 create mode 100644 build/lib/pipelines/automl/preprocess.py
 create mode 100644 build/lib/pipelines/automl/train.py
 create mode 100644 build/lib/pipelines/get_pipeline_definition.py
 create mode 100644 build/lib/pipel

To https://github.com/dgallitelli/autogluon-automl-build.git
 * [new branch]      main -> main
Flag --confirm has been deprecated, Pass any argument to skip confirmation prompt


...

https://github.com/dgallitelli/autogluon-automl-deploy
Created dgallitelli/autogluon-automl-deploy


Cloning into '/tmp/autogluon-automl-deploy'...


[main (root-commit) 4bb92c6] Initial seed code
 27 files changed, 972 insertions(+)
 create mode 100644 batch/__pycache__/build.cpython-312.pyc
 create mode 100644 batch/batch-transform-template.yml
 create mode 100644 batch/build.py
 create mode 100644 batch/buildspec.yml
 create mode 100644 batch/lambda/__pycache__/run_transform.cpython-312.pyc
 create mode 100644 batch/lambda/run_transform.py
 create mode 100644 batch/prod-config.json
 create mode 100644 batch/serve_batch.py
 create mode 100644 batch/staging-config.json
 create mode 100644 batch/test/buildspec.yml
 create mode 100644 batch/test/test.py
 create mode 100644 batch/tests/__pycache__/test_build.cpython-312-pytest-9.1.1.pyc
 create mode 100644 batch/tests/__pycache__/test_run_transform.cpython-312-pytest-9.1.1.pyc
 create mode 100644 batch/tests/test_build.py
 create mode 100644 batch/tests/test_run_transform.py
 create mode 100644 realtime/__pycache__/build.cpython-312.pyc
 create mode 100644 realtime/build.py
 create mo

Pushed seed code to dgallitelli/autogluon-automl-deploy


To https://github.com/dgallitelli/autogluon-automl-deploy.git
 * [new branch]      main -> main


## Find or create a CodeConnections connection tagged `sagemaker=true`

In [3]:
def find_sagemaker_connection():
    connections = codeconnections_client.list_connections()["Connections"]
    for conn in connections:
        if conn["ConnectionStatus"] != "AVAILABLE":
            continue
        tags = codeconnections_client.list_tags_for_resource(ResourceArn=conn["ConnectionArn"])["Tags"]
        if any(t["Key"] == "sagemaker" and t["Value"].lower() == "true" for t in tags):
            return conn["ConnectionArn"]
    return None

connection_arn = find_sagemaker_connection()

if connection_arn is None:
    create_response = codeconnections_client.create_connection(
        ProviderType="GitHub",
        ConnectionName=f"{PROJECT_NAME}-connection",
        Tags=[{"Key": "sagemaker", "Value": "true"}],
    )
    connection_arn = create_response["ConnectionArn"]
    print(f"Created connection {connection_arn} \u2014 go to the CodeConnections console to authorize it with GitHub, then re-run this cell.")
    print("Console: https://console.aws.amazon.com/codesuite/settings/connections")
else:
    print(f"Using existing connection: {connection_arn}")

Using existing connection: arn:aws:codeconnections:us-east-1:859755744029:connection/1144c539-6a48-4c3f-8d63-4ae68a0eabb1


## Poll until the connection is `AVAILABLE` (only needed if the previous cell just created one)

In [4]:
while True:
    status = codeconnections_client.get_connection(ConnectionArn=connection_arn)["Connection"]["ConnectionStatus"]
    print(f"Connection status: {status}")
    if status == "AVAILABLE":
        break
    print("Waiting 30s for you to authorize the connection in the console...")
    time.sleep(30)

Connection status: AVAILABLE


## Upload the CFN template to S3

In [5]:
TEMPLATE_KEY = "sagemaker-projects-templates/autogluon-automl-project-template.yaml"
s3_client = boto3.client("s3", region_name=REGION)
s3_client.upload_file("../cfn-templates/project-template.yaml", BUCKET, TEMPLATE_KEY)
template_url = f"https://{BUCKET}.s3.{REGION}.amazonaws.com/{TEMPLATE_KEY}"
print(f"Template uploaded: {template_url}")

Template uploaded: https://sagemaker-us-east-1-859755744029.s3.us-east-1.amazonaws.com/sagemaker-projects-templates/autogluon-automl-project-template.yaml


## Create the SageMaker Project

In [6]:
create_response = sm_client.create_project(
    ProjectName=PROJECT_NAME,
    ProjectDescription="AutoGluon AutoML CI/CD (build/train/evaluate/register + real-time and batch deploy)",
    TemplateProviders=[{
        "CfnTemplateProvider": {
            "TemplateName": "AutoGluonAutoMLProjectTemplate",
            "TemplateURL": template_url,
            "Parameters": [
                {"Key": "ModelBuildCodeRepositoryFullname", "Value": BUILD_REPO},
                {"Key": "ModelDeployCodeRepositoryFullname", "Value": DEPLOY_REPO},
                {"Key": "CodeConnectionArn", "Value": connection_arn},
            ],
        }
    }],
)
print(f"Project ARN: {create_response['ProjectArn']}")

Project ARN: arn:aws:sagemaker:us-east-1:859755744029:project/autogluon-automl


## Poll until `CreateCompleted`

In [7]:
while True:
    status = sm_client.describe_project(ProjectName=PROJECT_NAME)["ProjectStatus"]
    print(f"Project status: {status}")
    if status in ("CreateCompleted", "CreateFailed"):
        break
    time.sleep(30)

if status == "CreateFailed":
    raise Exception("Project creation failed \u2014 check the CloudFormation stack events in the console.")

project_id = sm_client.describe_project(ProjectName=PROJECT_NAME)["ProjectId"]
print(f"\nProject ready. ProjectId: {project_id}")
print(f"Build pipeline:        https://console.aws.amazon.com/codesuite/codepipeline/pipelines/sagemaker-{PROJECT_NAME}-{project_id}-modelbuild/view?region={REGION}")
print(f"Deploy (real-time):    https://console.aws.amazon.com/codesuite/codepipeline/pipelines/sagemaker-{PROJECT_NAME}-{project_id}-deploy-rt/view?region={REGION}")
print(f"Deploy (batch):        https://console.aws.amazon.com/codesuite/codepipeline/pipelines/sagemaker-{PROJECT_NAME}-{project_id}-deploy-batch/view?region={REGION}")

Project status: Pending


Project status: CreateInProgress


Project status: CreateInProgress


Project status: CreateInProgress


Project status: CreateInProgress


Project status: CreateCompleted



Project ready. ProjectId: p-z9hwuzukgdnk
Build pipeline:        https://console.aws.amazon.com/codesuite/codepipeline/pipelines/sagemaker-autogluon-automl-p-z9hwuzukgdnk-modelbuild/view?region=us-east-1
Deploy (real-time):    https://console.aws.amazon.com/codesuite/codepipeline/pipelines/sagemaker-autogluon-automl-p-z9hwuzukgdnk-deploy-rt/view?region=us-east-1
Deploy (batch):        https://console.aws.amazon.com/codesuite/codepipeline/pipelines/sagemaker-autogluon-automl-p-z9hwuzukgdnk-deploy-batch/view?region=us-east-1
